# LHRM DR1 — RUN_0_P1F_EXT1 — SAFE Kaggle notebook



**DIAGNOSTIC ONLY**, Variant L2, fresh 1000 optimizer steps. **Not** a 15-epoch MLTC baseline.

- Authentic `google-bert/bert-base-uncased` is downloaded from Hugging Face **during Cell 1** with Kaggle Internet **ON**, then read from a local folder only for the diagnostic. Zenodo AAPD **train/dev raw CSVs only**; strict CPU/GPU Step-0 cross-check.

- No Test read, model selection on Test, paper-hyperparameter modification, or FREEZE-B.

- Runtime BERT path is overridden in a **separate runtime config** only. The frozen YAML remains unchanged.

- The source CLI retains a legacy output filename `step300_diagnostics.json`; **the JSON step field is 1000**. The evidence ZIP contains explicit step-1000 documentation.



**Kaggle inputs:** `lhrm-bundle` (approved bundle) and `aapd-dataset` (approved original AAPD). No third-party BERT dataset. Set GPU=T4 and Internet=ON. **Do not toggle Kaggle Internet settings after download** (may restart runtime); Cell 1 sets the Hugging Face libraries to offline mode after acquisition.

Cells: 1=GPU check & pinned official-BERT download, then offline; 2=verify & stage inputs; 3=run diagnostic; 4=verify & export lightweight evidence.



Stop and share errors if a check fails. Do not bypass validation gates.


In [2]:
# Cell 1 — GPU gate, download ORIGINAL official pretrained BERT once, then offline mode
# Kaggle Settings: GPU=T4, Internet=ON. The download uses Hugging Face ONLY for official BERT.
import os,sys,time,json,hashlib,shutil,gc
from pathlib import Path
import torch
print('=== DR1 / Kaggle device + OFFICIAL BERT download ===',flush=True)
assert torch.cuda.is_available(),'CUDA GPU required — select Kaggle T4 or P100'
assert Path('/kaggle/input').is_dir(),'This notebook is for Kaggle'
print('GPU:',torch.cuda.get_device_name(0),'| torch:',torch.__version__,flush=True)
print('Working directory:',os.getcwd(),flush=True)

REPO='google-bert/bert-base-uncased'
BERT_DIR=Path('/kaggle/working/google_bert_bert_base_uncased_official')
SOURCE_META=Path('/kaggle/working/DR1_OFFICIAL_BERT_SOURCE.json')
WEIGHTS=('model.safetensors','pytorch_model.bin')
def sha256_bytes(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def complete_official_cache():
    return (BERT_DIR/'config.json').is_file() and (BERT_DIR/'vocab.txt').is_file() and \
        any((BERT_DIR/w).is_file() for w in WEIGHTS) and SOURCE_META.is_file()

if not complete_official_cache():
    assert not BERT_DIR.exists(), 'Partial BERT folder detected; stop to avoid silently reusing unverified weights: '+str(BERT_DIR)
    # Clear variables left behind by a previous notebook run; run this cell first in a fresh session.
    for key in ('HF_HUB_OFFLINE','TRANSFORMERS_OFFLINE'):
        os.environ.pop(key,None)
    print('Downloading official pretrained BERT from',REPO,'(Internet ON required)...',flush=True)
    from huggingface_hub import HfApi
    from transformers import BertModel,BertTokenizer
    revision=HfApi().model_info(REPO).sha  # immutable Hugging Face commit, logged for reproducibility
    assert revision and len(revision)==40, 'Could not resolve immutable official BERT revision'
    print('Pinned official BERT commit:',revision,flush=True)
    tokenizer=BertTokenizer.from_pretrained(REPO,revision=revision)
    assert tokenizer.vocab_size==30522, 'Unexpected BERT vocabulary'
    model=BertModel.from_pretrained(REPO,revision=revision)
    assert model.config.hidden_size==768 and model.config.vocab_size==30522
    tmp=BERT_DIR.with_name(BERT_DIR.name+'_DOWNLOADING')
    if tmp.exists():shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    try:
        model.save_pretrained(str(tmp),safe_serialization=True)
        tokenizer.save_pretrained(str(tmp))
        # Newer Transformers may save tokenizer.json without a separate vocab.txt.
        # Pin and copy the ORIGINAL vocab.txt bytes from the exact same official HF commit.
        from huggingface_hub import hf_hub_download
        official_vocab = Path(hf_hub_download(repo_id=REPO, filename='vocab.txt', revision=revision))
        shutil.copyfile(official_vocab, tmp/'vocab.txt')
        assert (tmp/'config.json').is_file() and (tmp/'vocab.txt').is_file()
        weight_file=next((tmp/w for w in WEIGHTS if (tmp/w).is_file()),None)
        assert weight_file is not None,'Missing BERT weights after saving'
        meta={'repo_id':REPO,'source_revision':revision,
              'saved_weights_file':weight_file.name,
              'saved_weights_sha256':sha256_bytes(weight_file),
              'bert_config_sha256':sha256_bytes(tmp/'config.json'),
              'vocab_sha256':sha256_bytes(tmp/'vocab.txt'),
              'vocab_size':30522,'hidden_size':768}
        BERT_DIR.parent.mkdir(parents=True,exist_ok=True)
        tmp.rename(BERT_DIR)
        SOURCE_META.write_text(json.dumps(meta,indent=2)+'\n',encoding='utf-8')
    finally:
        if tmp.exists():shutil.rmtree(tmp)
        del model,tokenizer
        gc.collect()
else:
    print('Official BERT already downloaded in this Kaggle runtime — checking cached hashes...',flush=True)

meta=json.loads(SOURCE_META.read_text('utf-8'))
assert meta.get('repo_id')==REPO and len(meta.get('source_revision',''))==40
assert sha256_bytes(BERT_DIR/meta['saved_weights_file'])==meta['saved_weights_sha256']
assert sha256_bytes(BERT_DIR/'config.json')==meta['bert_config_sha256']
assert sha256_bytes(BERT_DIR/'vocab.txt')==meta['vocab_sha256']
# Force all subsequent Transformers and Hugging Face loads to local files.
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['HF_HUB_OFFLINE']='1'
os.environ['HF_DATASETS_OFFLINE']='1'
print('OFFICIAL BERT DOWNLOADED & VERIFIED:',BERT_DIR,flush=True)
print('Source:',REPO,'@',meta['source_revision'],'| weights SHA256:',meta['saved_weights_sha256'],flush=True)
print('Hugging Face libraries are now OFFLINE. Do not change Kaggle Settings mid-session.',flush=True)


=== DR1 / Kaggle device + OFFICIAL BERT download ===
GPU: Tesla T4 | torch: 2.11.0+cu128
Working directory: /kaggle/working
Pinned official BERT commit: 86b5e0934494bd15c9632b12f734a8a67f723594


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

OFFICIAL BERT DOWNLOADED & VERIFIED: /kaggle/working/google_bert_bert_base_uncased_official
Source: google-bert/bert-base-uncased @ 86b5e0934494bd15c9632b12f734a8a67f723594 | weights SHA256: 5cc36ad3fafb0708f95cabab4966235f4662235815e77487b5e9c7d9692977d8
Hugging Face libraries are now OFFLINE. Do not change Kaggle Settings mid-session.


In [3]:
# Cell 2 — fail-closed bundle, BERT, and AAPD train/dev provenance
# This deliberately avoids `run_baseline.py prepare`: that command reads TEST.
from pathlib import Path
import os, json, hashlib, zipfile, shutil, re, io, time

WORK=Path('/kaggle/working')
INP=Path('/kaggle/input')
EXPECTED_BUNDLE_SHA='eebf2d9537c9949c3395b80eb707527f82ea08678ba2605bdf6c9fae0cf9a7ef'
EXPECTED_DATA_ZIP_SHA='e1b418427bcd2662f04c347099deb0d520b4066f782ed40c2a4aed15df694a08'
EXPECTED_CSV_SHA={
    'train':'e7e73064385f78fe092edd47d4e4059c35875a1ea2a95859ad3cde091ed9af95',
    'dev':'a85f612d4c1cb2be6d1de94cbcd9468cba97d221f826014264f4d279c82e220b',
}
EXPECTED_LABEL_SHA='a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b'
RUN_ID='RUN_0_P1F_EXT1'

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def sha_stream(f):
    h=hashlib.sha256()
    for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

# Locate the original exact ZIP or Kaggle's auto-extracted equivalent.
zip_candidates=[]; extracted_candidates=[]
for base,ds,fs in os.walk(INP):
    root=Path(base)
    if 'BUNDLE_MANIFEST.json' in fs and 'run_baseline.py' in fs:
        extracted_candidates.append(root)
    for name in fs:
        if name.lower().endswith('.zip') and 'lhrm' in name.lower():
            zip_candidates.append(root/name)
source=None
for zp in sorted(zip_candidates):
    if sha_file(zp)==EXPECTED_BUNDLE_SHA:
        source=('zip',zp);break
if source is None:
    for folder in sorted(extracted_candidates):
        try:
            manifest=json.loads((folder/'BUNDLE_MANIFEST.json').read_text('utf-8'))
            files=manifest['files']
            if all((folder/rel).is_file() and sha_file(folder/rel)==digest for rel,digest in files.items()):
                if manifest['bundle_name']=='lhrm_gpu_bundle_20261010_003953.zip':
                    source=('directory',folder);break
        except (OSError,KeyError,ValueError):pass
if source is None:
    raise FileNotFoundError('Exact approved bundle not found. Attach bundle ZIP as Kaggle dataset; unzipped bundle is also supported.')
print('Bundle source VERIFIED:',*source,flush=True)
if source[0]=='zip':
    with zipfile.ZipFile(source[1]) as z:
        for n in z.namelist():
            if Path(n).is_absolute() or '..' in Path(n).parts:
                raise ValueError('Unsafe bundle member: '+n)
        z.extractall(WORK)
else:
    for base,ds,fs in os.walk(source[1]):
        for n in fs:
            src=Path(base)/n
            rel=src.relative_to(source[1]);dest=WORK/rel
            dest.parent.mkdir(parents=True,exist_ok=True)
            if src.resolve()!=dest.resolve():shutil.copy2(src,dest)
manifest=json.loads((WORK/'BUNDLE_MANIFEST.json').read_text('utf-8'))
assert manifest.get('label_order_sha256')==EXPECTED_LABEL_SHA
for rel,dig in manifest['files'].items():
    assert sha_file(WORK/rel)==dig, 'Bundle component differs: '+rel
print('Bundle file integrity: PASS ({} files)'.format(len(manifest['files'])),flush=True)

# Use ONLY the official original BERT downloaded by Cell 1, not arbitrary Kaggle models.
# Source repo/revision and local file hashes were recorded and verified in Cell 1.
BERT_PATH=WORK/'google_bert_bert_base_uncased_official'
source_meta_file=WORK/'DR1_OFFICIAL_BERT_SOURCE.json'
assert source_meta_file.is_file() and BERT_PATH.is_dir(), 'Run Cell 1 with Internet ON to download official BERT first'
source_meta=json.loads(source_meta_file.read_text('utf-8'))
assert source_meta['repo_id']=='google-bert/bert-base-uncased'
assert sha_file(BERT_PATH/source_meta['saved_weights_file'])==source_meta['saved_weights_sha256'], 'BERT weights hash mismatch'
assert sha_file(BERT_PATH/'config.json')==source_meta['bert_config_sha256']
assert sha_file(BERT_PATH/'vocab.txt')==source_meta['vocab_sha256']
from transformers import BertTokenizer, BertModel
bert_tokenizer=BertTokenizer.from_pretrained(str(BERT_PATH),local_files_only=True)
assert bert_tokenizer.vocab_size==30522,'Unexpected BERT vocabulary size'
bert_cfg=json.loads((BERT_PATH/'config.json').read_text())
assert bert_cfg.get('hidden_size')==768 and bert_cfg.get('vocab_size')==30522
print('BERT candidate path:',BERT_PATH,flush=True)
print('BERT identity: subject to mandatory seven-field Step-0 cross-check, not inferred from path alone.',flush=True)

# Find authoritative raw Zenodo CSV files (original bytes) in ZIP or extracted folder.
# Do not open/load/predict/test on test.csv in DR1.
csv_dirs=[];data_zips=[]
for base,ds,fs in os.walk(INP):
    root=Path(base)
    if 'train.csv' in fs and 'dev.csv' in fs:csv_dirs.append(root)
    for name in fs:
        if name.lower().endswith('.zip') and ('aapd' in str(root/name).lower()):data_zips.append(root/name)
valid=[]
for zp in sorted(data_zips):
    try:
        with zipfile.ZipFile(zp) as z:
            members={}
            for split in ('train','dev'):
                choices=[x for x in z.namelist() if x.replace('\\','/').endswith('/'+split+'.csv') or x==split+'.csv']
                if len(choices)!=1:raise ValueError('Ambiguous AAPD ZIP members')
                members[split]=choices[0]
            hashes={s:sha_stream(z.open(members[s])) for s in ('train','dev')}
        if hashes==EXPECTED_CSV_SHA:
            valid.append(('zip',zp,members,sha_file(zp)))
    except (ValueError,zipfile.BadZipFile,OSError):pass
for dr in sorted(csv_dirs):
    hashes={s:sha_file(dr/(s+'.csv')) for s in ('train','dev')}
    if hashes==EXPECTED_CSV_SHA:valid.append(('directory',dr,None,None))
if not valid:raise RuntimeError('AAPD train/dev source not found or SHA256 MISMATCH. Do not alter CSVs. Attach original Zenodo AAPD dataset.')
valid.sort(key=lambda x:(0 if x[3]==EXPECTED_DATA_ZIP_SHA else 1,x[0],str(x[1])))
kind,data_path,members,outer_hash=valid[0]
print('AAPD source: VERIFIED',kind,data_path,flush=True)
print('AAPD raw SHA256 verified: train={}, dev={}'.format(EXPECTED_CSV_SHA['train'][:16],EXPECTED_CSV_SHA['dev'][:16]),flush=True)
print('Original AAPD ZIP SHA256 matched:',outer_hash==EXPECTED_DATA_ZIP_SHA if outer_hash is not None else 'Not available; individual raw CSV hashes match',flush=True)

# Stage unchanged Train/Dev CSV bytes for the existing canonical project loader.
# No test file is extracted, read or prepared.
STAGED=WORK/'AAPD_DR1_TRAIN_DEV_ONLY.zip'
with zipfile.ZipFile(STAGED,'w',compression=zipfile.ZIP_STORED) as out:
    for split in ('train','dev'):
        with out.open('AAPD/'+split+'.csv','w') as dest:
            if kind=='zip':
                with zipfile.ZipFile(data_path) as inz, inz.open(members[split]) as src:
                    shutil.copyfileobj(src,dest,1024*1024)
            else:
                with open(data_path/(split+'.csv'),'rb') as src:
                    shutil.copyfileobj(src,dest,1024*1024)
with zipfile.ZipFile(STAGED) as staged:
    assert all(sha_stream(staged.open('AAPD/'+s+'.csv'))==EXPECTED_CSV_SHA[s] for s in ('train','dev'))

sys.path.insert(0,str(WORK/'src'))
from lhrm_baseline.data import load_aapd_split,save_processed
from lhrm_baseline.labels import build_all_label_texts
processed=WORK/'data/processed/AAPD'
processed.mkdir(parents=True,exist_ok=True)
label_order=None
for split,expected_size in (('train',53840),('dev',1000)):
    texts,labels,cols=load_aapd_split(str(STAGED),split)
    assert len(texts)==expected_size and labels.shape==(expected_size,54)
    if label_order is None:label_order=cols
    else:assert cols==label_order,'Train/dev label order mismatch'
    save_processed(str(processed),split,texts,labels,cols)
    print('Prepared',split,expected_size,'docs 54 labels, no Test',flush=True)
label_sha=hashlib.sha256(json.dumps(label_order).encode()).hexdigest()
assert label_sha==EXPECTED_LABEL_SHA,'AAPD label order SHA256 mismatch'
defs_path=WORK/'Label_Definition/arxiv_labels_definitions.json'
label_texts=build_all_label_texts(label_order,str(defs_path),dataset='aapd')
assert len(label_texts)==54 and all(x.strip() for x in label_texts)
(processed/'label_texts.json').write_text(json.dumps(label_texts,indent=2,ensure_ascii=False),encoding='utf-8')
assert not list(processed.glob('test_*')),'This DR1 directory must not contain test data'

# The verified 200 preregistered validation IDs are already inside the bundle.
ids_file=WORK/'experiments/RUN_0/run0_val_sample_ids.json'
assert ids_file.is_file()
ids=json.loads(ids_file.read_text())
assert len(ids)==200 and len(set(ids))==200 and all(0<=i<1000 for i in ids)
print('Registered validation IDs SHA256:',sha_file(ids_file),flush=True)

# Preserve the frozen config. Override only BERT's absolute offline location.
orig=WORK/'configs/final/aapd_baseline.yaml'
cfg=orig.read_text(encoding='utf-8')
needle='bert_model: bert-base-uncased'
assert cfg.count(needle)==1,'Unexpected frozen baseline config format'
runtime=WORK/'configs/final/aapd_dr1_runtime.yaml'
runtime.write_text(cfg.replace(needle,'bert_model: '+str(BERT_PATH)),encoding='utf-8')
from run_baseline import load_yaml_config
before=load_yaml_config(str(orig));after=load_yaml_config(str(runtime))
assert {k:(before[k],after[k]) for k in before if before[k]!=after[k]}=={
    'bert_model':('bert-base-uncased',str(BERT_PATH))},'Unexpected change to research configuration'

PROVENANCE={
    'run_id':RUN_ID,'purpose':'DIAGNOSTIC_ONLY','steps':1000,'candidates':['L2'],
    'original_bundle_sha256':EXPECTED_BUNDLE_SHA,'bundle_source':str(source[1]),
    'frozen_config_sha256':sha_file(orig),'runtime_config_sha256':sha_file(runtime),
    'runtime_config_change':'bert_model local absolute path ONLY',
    'bert_local_path':str(BERT_PATH),
    'bert_hf_repo':source_meta['repo_id'],'bert_hf_commit':source_meta['source_revision'],
    'bert_saved_weights_sha256':source_meta['saved_weights_sha256'],
    'aapd_source':str(data_path),'aapd_source_kind':kind,
    'zenodo_original_zip_sha256_if_accessible':outer_hash,
    'raw_train_dev_sha256':EXPECTED_CSV_SHA,
    'staged_train_dev_zip_sha256':sha_file(STAGED),
    'label_order_sha256':label_sha,'val_ids_file_sha256':sha_file(ids_file),
    'test_split_loaded':False,'gpu_name':torch.cuda.get_device_name(0),
    'source_cli_legacy_diag_filename':'variant_L2/step300_diagnostics.json (contains step=1000)'
}
(WORK/'DR1_KAGGLE_INPUT_PROVENANCE.json').write_text(json.dumps(PROVENANCE,indent=2),encoding='utf-8')
print('\n=== INPUT GATES PASSED: L2 / 1000 steps / train+dev only ===',flush=True)


Bundle source VERIFIED: directory /kaggle/input/datasets/nouraelsaman/lhrm-bundle
Bundle file integrity: PASS (23 files)
BERT candidate path: /kaggle/working/google_bert_bert_base_uncased_official
BERT identity: subject to mandatory seven-field Step-0 cross-check, not inferred from path alone.
AAPD source: VERIFIED directory /kaggle/input/datasets/nouraelsaman/aapd-dataset/AAPD
AAPD raw SHA256 verified: train=e7e73064385f78fe, dev=a85f612d4c1cb2be
Original AAPD ZIP SHA256 matched: Not available; individual raw CSV hashes match
Prepared train 53840 docs 54 labels, no Test
Prepared dev 1000 docs 54 labels, no Test
Registered validation IDs SHA256: 25912e362b0180f0b47a694ecf1d29c35f4140c87375a8053a62fee6850df0a8

=== INPUT GATES PASSED: L2 / 1000 steps / train+dev only ===


In [4]:
# Cell 3 — DR1 GPU execution. Fail closed; uses trusted bundle CLI as supplied.
import os,sys,subprocess,time
from pathlib import Path
WORK=Path('/kaggle/working')
assert (WORK/'DR1_KAGGLE_INPUT_PROVENANCE.json').is_file(),'Run Cell 2 first'
RUN_ID='RUN_0_P1F_EXT1'
cmd=[sys.executable,'-u',str(WORK/'run_baseline.py'),'run0',
    '--config',str(WORK/'configs/final/aapd_dr1_runtime.yaml'),
    '--run_id',RUN_ID,'--steps','1000','--candidates','L2']
print('=== START DR1 DIAGNOSTIC_ONLY (NOT FULL EPOCH TRAINING) ===',flush=True)
print(' '.join(cmd),flush=True)
env=os.environ.copy()
env.update({'TRANSFORMERS_OFFLINE':'1','HF_HUB_OFFLINE':'1',
            'HF_DATASETS_OFFLINE':'1','PYTHONUNBUFFERED':'1'})
start=time.monotonic()
with subprocess.Popen(cmd,cwd=str(WORK),env=env,stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT,bufsize=1,text=True) as p:
    for line in p.stdout:
        print(line,end='',flush=True)
    code=p.wait()
print('DR1 wall time (seconds):',round(time.monotonic()-start,1),flush=True)
if code!=0:raise RuntimeError('DR1 failed (exit code {}). Do NOT run evidence export as if successful.'.format(code))
print('DR1 CLI completed — run Cell 4 for strict result audit.',flush=True)


=== START DR1 DIAGNOSTIC_ONLY (NOT FULL EPOCH TRAINING) ===
/usr/bin/python3 -u /kaggle/working/run_baseline.py run0 --config /kaggle/working/configs/final/aapd_dr1_runtime.yaml --run_id RUN_0_P1F_EXT1 --steps 1000 --candidates L2
RUN_0 -- AAPD Architecture Screening
Candidates: ['L2']   Steps: 1000   Seed: 42
Validation sample: 200 docs (IDs copied from registered RUN_0)
Label-order hash: OK (a3d662f39202...)
Labels: 54
  Prior P@1 on 200 val docs (seeded tie-break): 63/200 = 0.3150

--------------------------------------------------
Candidate: Variant L2
--------------------------------------------------

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2848.27it/s]
  Step 0: mean_max_alpha=0.029699 one_hot_frac=0.0000 P@1=0/200 S_mean=6.04
  Preflight cross-check PASSED (7 fields within tolerance).
  Step   50/1000: loss=0.0152
  Step  100/1000: loss=0.0182
  Step  150/1000: loss=0.0190
  Checkpoint saved: step_150.pt
  Step  200/1000: loss=0.0136
  Step  250/1000: loss=0.015

In [5]:
# Cell 4 — Verify genuine step-1000 results; package SMALL evidence only
import json,zipfile,hashlib,os,time
from pathlib import Path
WORK=Path('/kaggle/working');RUN_ID='RUN_0_P1F_EXT1'
EXP=WORK/'experiments'/RUN_ID
assert (EXP/'run0_decision.json').is_file(),'Run did not complete; no evidence export'
required=['run0_decision.json','env.json','resolved_config.json',
    'run0_val_sample_ids.json','variant_L2/train.log',
    'variant_L2/step0_diagnostics.json',
    'variant_L2/step300_diagnostics.json',
    'variant_L2/val_sample_predictions.json']
for rel in required:
    assert (EXP/rel).is_file(),'Missing required DR1 evidence: '+rel
assert not (EXP/'variant_L').exists(),'Wrong candidate present: L'
decision=json.loads((EXP/'run0_decision.json').read_text())
diag=json.loads((EXP/'variant_L2/step300_diagnostics.json').read_text())
resolved=json.loads((EXP/'resolved_config.json').read_text())
assert diag.get('step')==1000,'Diagnostics did not run at step 1000'
assert resolved.get('n_steps')==1000 and resolved.get('candidates')==['L2']
assert resolved.get('n_val_docs')==200 and resolved.get('prior_p_at_1_hits')==63
assert decision.get('candidate_results',{}).keys()=={'L2'},'Unexpected candidates in decision'
result=decision['candidate_results']['L2']
assert all(k in result for k in ('V1','V2','V3','V4','V5','p_at_1_hits','prior_p_at_1_hits'))
assert result['prior_p_at_1_hits']==63
v4_n=sum(float(x)>0.01 for x in diag.get('per_label_logit_std',[]))
assert len(diag.get('per_label_logit_std',[]))==54,'Missing logit std per label'
assert (v4_n>=28)==bool(result['V4']), 'V4 result does not match strict >50% threshold'
summary={
    'run_id':RUN_ID,'run_type':'DIAGNOSTIC_ONLY',
    'diagnostics_actual_step':1000,
    'legacy_source_filename':'variant_L2/step300_diagnostics.json',
    'source_filename_note':'Legacy basename unchanged in CLI; JSON step field is 1000',
    'v4_labels_passing_std_threshold':v4_n,
    'v4_required_labels':28,
    'v5_model_hits':result['p_at_1_hits'],'v5_prior_hits':63,
    'criteria':{k:bool(result[k]) for k in ('V1','V2','V3','V4','V5')},
    'viable_all_criteria':bool(result.get('viable',False)),
    'script_decision_status':decision.get('status'),
    'script_chosen_variant':decision.get('chosen_variant'),
    'note':'NO automatic FREEZE-B; human evidence review and explicit gate are required.'
}
print('=== DR1 REAL RESULTS ===',flush=True)
print(json.dumps(summary,indent=2),flush=True)
# No checkpoint inside lightweight evidence ZIP; include all small run outputs and input provenance.
OUT=WORK/(RUN_ID+'_EVIDENCE_'+time.strftime('%Y%m%d_%H%M%S')+'.zip')
written=[]
with zipfile.ZipFile(OUT,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for path in sorted(EXP.rglob('*')):
        if not path.is_file() or path.suffix.lower() in ('.pt','.pth','.ckpt','.bin','.safetensors'):
            continue
        rel='RUN_0_P1F_EXT1/'+path.relative_to(EXP).as_posix()
        z.write(path,rel)
        written.append(rel)
    z.write(WORK/'DR1_KAGGLE_INPUT_PROVENANCE.json','DR1_KAGGLE_INPUT_PROVENANCE.json')
    z.write(WORK/'DR1_OFFICIAL_BERT_SOURCE.json','DR1_OFFICIAL_BERT_SOURCE.json')
    z.write(WORK/'configs/final/aapd_dr1_runtime.yaml','aapd_dr1_runtime.yaml')
    z.writestr('DR1_STEP1000_SUMMARY.json',json.dumps(summary,indent=2))
    # Offer correct step-1000 alias INSIDE evidence ZIP, without modifying any source artifact.
    z.write(EXP/'variant_L2/step300_diagnostics.json',
            'RUN_0_P1F_EXT1/variant_L2/step1000_diagnostics.json')
print('Evidence ZIP:',OUT,flush=True)
print('SHA256:',sha_file(OUT),flush=True)
print('Size MB:',round(OUT.stat().st_size/1e6,2),'files:',len(written)+5,flush=True)
print('Download evidence ZIP only. Do NOT call FREEZE-B until audit.',flush=True)


=== DR1 REAL RESULTS ===
{
  "run_id": "RUN_0_P1F_EXT1",
  "run_type": "DIAGNOSTIC_ONLY",
  "diagnostics_actual_step": 1000,
  "legacy_source_filename": "variant_L2/step300_diagnostics.json",
  "source_filename_note": "Legacy basename unchanged in CLI; JSON step field is 1000",
  "v4_labels_passing_std_threshold": 49,
  "v4_required_labels": 28,
  "v5_model_hits": 63,
  "v5_prior_hits": 63,
  "criteria": {
    "V1": true,
    "V2": true,
    "V3": true,
    "V4": true,
    "V5": false
  },
  "viable_all_criteria": false,
  "script_decision_status": "EXTEND",
  "script_chosen_variant": null,
  "note": "NO automatic FREEZE-B; human evidence review and explicit gate are required."
}
Evidence ZIP: /kaggle/working/RUN_0_P1F_EXT1_EVIDENCE_20261009_222831.zip
SHA256: 806fce2e7f001bcabcd4d23f451b7a80f44dd5b5530f38a53d239a66d6544329
Size MB: 0.13 files: 14
Download evidence ZIP only. Do NOT call FREEZE-B until audit.
